# Expert 03 — Design Patterns (Pythonic)


## 1. Singleton

In [ ]:
# Method 1: Metaclass
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self):
        self.debug = False
        self.db_url = 'sqlite:///app.db'

c1 = Config()
c2 = Config()
print(c1 is c2)  # True

# Method 2: Module-level (Pythonic — modules are singletons)
# config.py:
# DEBUG = False
# DB_URL = 'sqlite:///app.db'
# import config; config.DEBUG = True

## 2. Factory Pattern

In [ ]:
from abc import ABC, abstractmethod

class Serializer(ABC):
    @abstractmethod
    def serialize(self, data: dict) -> str: ...

    @abstractmethod
    def deserialize(self, text: str) -> dict: ...

class JSONSerializer(Serializer):
    def serialize(self, data):
        import json; return json.dumps(data)
    def deserialize(self, text):
        import json; return json.loads(text)

class CSVSerializer(Serializer):
    def serialize(self, data):
        return ','.join(f'{k}={v}' for k, v in data.items())
    def deserialize(self, text):
        return dict(pair.split('=') for pair in text.split(','))

# Factory function (Pythonic — no Factory class needed)
_SERIALIZERS = {'json': JSONSerializer, 'csv': CSVSerializer}

def get_serializer(fmt: str) -> Serializer:
    cls = _SERIALIZERS.get(fmt.lower())
    if cls is None:
        raise ValueError(f'Unknown format: {fmt}. Choose from {list(_SERIALIZERS)}')
    return cls()

data = {'name': 'Alice', 'age': '30'}
for fmt in ['json', 'csv']:
    s = get_serializer(fmt)
    text = s.serialize(data)
    back = s.deserialize(text)
    print(f'{fmt}: {text!r} -> {back}')

## 3. Observer Pattern

In [ ]:
from typing import Callable, Any
from collections import defaultdict

class EventBus:
    """Simple publish-subscribe event bus."""

    def __init__(self):
        self._handlers: dict[str, list[Callable]] = defaultdict(list)

    def subscribe(self, event: str, handler: Callable) -> None:
        self._handlers[event].append(handler)

    def unsubscribe(self, event: str, handler: Callable) -> None:
        self._handlers[event].remove(handler)

    def publish(self, event: str, **data: Any) -> None:
        for handler in self._handlers[event]:
            handler(**data)

    def on(self, event: str):
        """Decorator for subscribing."""
        def decorator(func):
            self.subscribe(event, func)
            return func
        return decorator


bus = EventBus()

@bus.on('user.created')
def send_welcome_email(user_id, email, **_):
    print(f'Sending welcome email to {email}')

@bus.on('user.created')
def create_profile(user_id, **_):
    print(f'Creating profile for user {user_id}')

@bus.on('user.deleted')
def cleanup_data(user_id, **_):
    print(f'Cleaning up data for user {user_id}')

bus.publish('user.created', user_id=1, email='alice@example.com')
print()
bus.publish('user.deleted', user_id=1)

## 4. Strategy Pattern

In [ ]:
from typing import Protocol

# Python 3.8+ Protocol — structural subtyping (duck typing)
class SortStrategy(Protocol):
    def sort(self, data: list) -> list: ...

class BubbleSort:
    def sort(self, data: list) -> list:
        data = data.copy()
        n = len(data)
        for i in range(n):
            for j in range(n - i - 1):
                if data[j] > data[j+1]:
                    data[j], data[j+1] = data[j+1], data[j]
        return data

class QuickSort:
    def sort(self, data: list) -> list:
        if len(data) <= 1:
            return data
        pivot = data[len(data) // 2]
        left  = [x for x in data if x < pivot]
        mid   = [x for x in data if x == pivot]
        right = [x for x in data if x > pivot]
        return self.sort(left) + mid + self.sort(right)

class Sorter:
    def __init__(self, strategy: SortStrategy):
        self.strategy = strategy

    def sort(self, data: list) -> list:
        return self.strategy.sort(data)

data = [64, 34, 25, 12, 22, 11, 90]
for strategy in [BubbleSort(), QuickSort()]:
    sorter = Sorter(strategy)
    print(f'{strategy.__class__.__name__}: {sorter.sort(data)}')

## 5. Decorator Pattern

In [ ]:
import time
from functools import wraps

# Timing decorator
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'{func.__name__} took {elapsed*1000:.2f}ms')
        return result
    return wrapper

# Retry decorator
def retry(max_attempts=3, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    print(f'Attempt {attempt} failed: {e}, retrying...')
        return wrapper
    return decorator

# Cache decorator
def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache
    wrapper.cache_clear = cache.clear
    return wrapper

@timed
@memoize
def fib(n):
    if n <= 1: return n
    return fib(n-1) + fib(n-2)

print(fib(30))
print(fib(30))  # cached

## 6. Context Manager Pattern

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label=''):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label}: {elapsed*1000:.2f}ms')

@contextmanager
def transaction(db):
    """Simulate DB transaction."""
    print('BEGIN TRANSACTION')
    try:
        yield db
        print('COMMIT')
    except Exception as e:
        print(f'ROLLBACK: {e}')
        raise

with timer('list comprehension'):
    result = [x**2 for x in range(100_000)]

with timer('generator sum'):
    result = sum(x**2 for x in range(100_000))